In [ ]:
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display

import predictive_coding as pc

SEED = 42
rng = np.random.default_rng(SEED)   # NumPy: data and starting angles
torch.manual_seed(SEED)             # PyTorch: initial weights
torch.set_num_threads(1)            # a tiny network: extra threads only add overhead

# runs/ is listed in .gitignore, so nothing written here gets committed.
RUN_DIR = Path("../runs/planar_arm_tutorial")
RUN_DIR.mkdir(parents=True, exist_ok=True)

L1, L2 = 1.0, 1.0                   # segment lengths

In [ ]:
def forward_kinematics(theta1, theta2):
    """Joint angles (radians) -> (elbow, tip) positions. Works on numbers and on arrays."""
    elbow = np.stack([L1 * np.cos(theta1), L1 * np.sin(theta1)], axis=-1)
    tip = elbow + np.stack([L2 * np.cos(theta1 + theta2), L2 * np.sin(theta1 + theta2)], axis=-1)
    return elbow, tip


def draw_arm(ax, theta1, theta2, color="C0", alpha=1.0, label=None):
    """Draw the two segments: shoulder (origin) -> elbow -> tip."""
    elbow, tip = forward_kinematics(theta1, theta2)
    ax.plot([0, elbow[0], tip[0]], [0, elbow[1], tip[1]], "-o",
            color=color, lw=3, ms=5, alpha=alpha, label=label)


def arm_axes(ax, title=""):
    """Square axes with the reachable disk (radius L1 + L2) drawn dashed."""
    ax.add_patch(plt.Circle((0, 0), L1 + L2, fill=False, ls="--", color="gray"))
    ax.set_xlim(-2.3, 2.3)
    ax.set_ylim(-2.3, 2.3)
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)
    ax.set_title(title)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

arm_axes(axes[0], "a few postures")
for t1, t2, color in [(0.3,0.8, "C0"), (2.0, -1.2, "C1"), (-1.0, 2.5, "C2")]:
    draw_arm(axes[0], t1, t2, color=color, label=f"θ1={t1}, θ2={t2}")
axes[0].legend(fontsize=8)

# With L1 = L2, the postures (θ1, θ2) and (θ1 + θ2, -θ2) put the tip in the same place.
t1, t2 = 0.2, 1.4
arm_axes(axes[1], "one tip, two postures")
draw_arm(axes[1], t1, t2, color="C0", label=f"({t1}, {t2})")
draw_arm(axes[1], t1 + t2, -t2, color="C3", label=f"({t1 + t2:.1f}, {-t2})")
axes[1].legend(fontsize=8)
plt.show()

print("tips:", forward_kinematics(t1, t2)[1], forward_kinematics(t1 + t2, -t2)[1])

In [ ]:
N = 10_000
angles = rng.uniform(-np.pi, np.pi, size=(N, 2))          # (θ1, θ2)
_, tips = forward_kinematics(angles[:, 0], angles[:, 1])   # (x, y)

n_train = 9_000
q_train = torch.tensor(angles[:n_train], dtype=torch.float32)
xy_train = torch.tensor(tips[:n_train], dtype=torch.float32)
q_test = torch.tensor(angles[n_train:], dtype=torch.float32)
xy_test = torch.tensor(tips[n_train:], dtype=torch.float32)

loader = DataLoader(TensorDataset(q_train, xy_train), batch_size=128, shuffle=True,
                    generator=torch.Generator().manual_seed(SEED))
print(q_train.shape, xy_train.shape, q_test.shape, xy_test.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
color = angles[:3000, 1]   # colour every sample by its elbow angle θ2

axes[0].scatter(angles[:3000, 0], angles[:3000, 1], s=3, c=color)
axes[0].set(title="angle space (uniform)", xlabel="θ1", ylabel="θ2", aspect="equal")

arm_axes(axes[1], "tip positions (not uniform)")
axes[1].scatter(tips[:3000, 0], tips[:3000, 1], s=3, c=color, cmap="twilight")
plt.show()

In [ ]:
class ArmNet(nn.Module):
    """Predicts the tip position (x, y) from the joint angles (θ1, θ2)."""

    def __init__(self, hidden=32):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(2, hidden),
            pc.PCLayer(is_keep_energy_per_datapoint=True), 
            nn.Tanh(),
            nn.Linear(hidden, hidden), 
            pc.PCLayer(is_keep_energy_per_datapoint=True), 
            nn.Tanh(),
            nn.Linear(hidden, 2),
        )
        self.pcs = [self.layers[1], self.layers[4]]   # the two PCLayers: z1 and z2

    def forward(self, q):
        return self.layers(q)


def output_energy(output, target):
    """The output term of the energy, 0.5 * ||y_hat - y||^2, summed over the batch."""
    return 0.5 * (output - target).pow(2).sum()


model = ArmNet()
print(model)
print("weights:", sum(p.numel() for p in model.parameters()))

In [ ]:
T = 30   # steps per batch: 29 inference steps (update_x_at) + 1 weight step (update_p_at)

trainer = pc.PCTrainer(
    model,
    T=T,
    optimizer_x_fn=optim.SGD, optimizer_x_kwargs={"lr": 0.1},      # inference: z1, z2
    update_x_at=list(range(T - 1)),                                # steps 0..28
    optimizer_p_fn=optim.Adam, optimizer_p_kwargs={"lr": 1e-3},    # learning: W, b
    update_p_at="last",                                            # step 29
    update_p_at_early_stop=False,
    plot_progress_at=[],
)


def predict(model, q):
    """One feed-forward pass (PCLayers transparent in eval mode). Returns a NumPy array."""
    model.eval()
    with torch.no_grad():
        out = model(q).numpy()
    model.train()
    return out


def rmse(model, q, xy):
    """Root mean squared distance between predicted and true tips."""
    dist = np.linalg.norm(predict(model, q) - xy.numpy(), axis=1)
    return float(np.sqrt(np.mean(dist ** 2)))

In [ ]:
epochs = 60
snapshot_epochs = [0, 1, 3, 10, 60]
snapshots = {}
history = {"train": [], "test": []}
first_batch_trace = None

model.train()   # PCLayers must be in train mode while training (the trainer asserts this)
for epoch in range(epochs + 1):
    if epoch > 0:
        for i, (q_batch, xy_batch) in enumerate(loader):
            record = (epoch == 1 and i == 0)   # keep every inference step, only once
            result = trainer.train_on_batch(
                inputs=q_batch,
                loss_fn=output_energy,
                loss_fn_kwargs={"target": xy_batch},
                is_return_results_every_t=record,
            )
            if record:
                first_batch_trace = result

    history["train"].append(rmse(model, q_train, xy_train))
    history["test"].append(rmse(model, q_test, xy_test))
    if epoch in snapshot_epochs:
        snapshots[epoch] = predict(model, q_test)
    if epoch % 10 == 0:
        print(f"epoch {epoch:2d}: train RMSE {history['train'][-1]:.3f}   test RMSE {history['test'][-1]:.3f}")

In [ ]:
plt.semilogy(history["train"], label="train")
plt.semilogy(history["test"], label="test")
plt.xlabel("epoch")
plt.ylabel("RMSE of the tip position")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(snapshots) + 1, figsize=(3.2 * (len(snapshots) + 1), 3.6))
color = angles[n_train:, 1]
for ax, (epoch, pred) in zip(axes, snapshots.items()):
    arm_axes(ax, f"epoch {epoch}")
    ax.scatter(pred[:, 0], pred[:, 1], s=3, c=color, cmap="twilight")
arm_axes(axes[-1], "truth")
axes[-1].scatter(tips[n_train:, 0], tips[n_train:, 1], s=3, c=color, cmap="twilight")
plt.tight_layout()
plt.show()

In [ ]:
grid = np.linspace(-np.pi, np.pi, 201)
T1, T2 = np.meshgrid(grid, grid)
q_grid = np.stack([T1.ravel(), T2.ravel()], axis=1)

_, true_grid = forward_kinematics(q_grid[:, 0], q_grid[:, 1])
pred_grid = predict(model, torch.tensor(q_grid, dtype=torch.float32))
error_map = np.linalg.norm(pred_grid - true_grid, axis=1).reshape(T1.shape)

plt.imshow(error_map, origin="lower", extent=[-np.pi, np.pi, -np.pi, np.pi], cmap="magma")
plt.colorbar(label="tip error")
plt.xlabel("θ1")
plt.ylabel("θ2")
plt.title("prediction error over angle space")
plt.show()

In [ ]:
steps = np.arange(T)
batch_size = 128
plt.plot(steps, np.array(first_batch_trace["overall"]) / batch_size, "k", lw=2, label="total E")
plt.plot(steps, np.array(first_batch_trace["loss"]) / batch_size, label="output: ½‖ŷ − y‖²")
plt.plot(steps, np.array(first_batch_trace["energy"]) / batch_size, label="hidden: ½‖z1 − μ1‖² + ½‖z2 − μ2‖²")
plt.xlabel("inference step t (weights fixed until the last one)")
plt.ylabel("energy per sample")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
def weights_only(state_dict):
    """Drop the PCLayer latents (keys ending in '._x'): they belong to the last batch, not to the model."""
    return {k: v for k, v in state_dict.items() if not k.endswith("._x")}


ckpt_path = RUN_DIR / "armnet.pt"
torch.save(weights_only(model.state_dict()), ckpt_path)

arm_model = ArmNet()
print(arm_model.load_state_dict(torch.load(ckpt_path)))

# Freeze the weights: while reaching, only the angles and the latents change.
for p in arm_model.parameters():
    p.requires_grad_(False)

In [ ]:
def reach(model, targets, q_start, steps=20_000, lr_q=0.05, lr_z=0.1, record_every=50):
    """
    Clamp target tip positions at the output and let the angles q (and the latents) settle.

    targets: list of (x, y), one per arm
    q_start: one (θ1, θ2) shared by all arms, or a list with one per arm
    Returns a dict of NumPy arrays recorded every `record_every` steps.
    """
    model.train()                                   # PCLayers use their latents z
    target = torch.tensor(targets, dtype=torch.float32)
    q_start = np.broadcast_to(np.asarray(q_start, dtype=float), (len(targets), 2))

    # (a) The angles are a free variable: an nn.Parameter that the optimiser can move.
    q = nn.Parameter(torch.tensor(q_start, dtype=torch.float32))

    # (b) Initialise z1, z2 with one feed-forward pass from the starting angles (z := mu).
    for layer in model.pcs:
        layer.set_is_sample_x(True)
    model(q)
    z = [layer.get_x() for layer in model.pcs]

    # (c) Plain gradient descent, with one step size for the angles and one for the latents.
    opt = optim.SGD([{"params": [q], "lr": lr_q}, {"params": z, "lr": lr_z}])

    rec = {"step": [], "q": [], "y_relaxed": [], "E_hidden": [], "E_out": []}
    for step in range(steps + 1):
        opt.zero_grad()
        y_relaxed = model(q)                        # y_hat = W3 f(z2) + b3
        E = sum(layer.energy() for layer in model.pcs) + output_energy(y_relaxed, target)

        if step % record_every == 0:
            rec["step"].append(step)
            rec["q"].append(q.detach().numpy().copy())
            rec["y_relaxed"].append(y_relaxed.detach().numpy().copy())
            rec["E_hidden"].append(sum(layer.energy_per_datapoint() for layer in model.pcs).detach().numpy()[:, 0])
            rec["E_out"].append(0.5 * (y_relaxed - target).pow(2).sum(dim=1).detach().numpy())
        if step == steps:
            break

        E.backward()                                # dE/dq, dE/dz1, dE/dz2 (Section 1)
        opt.step()                                  # variable <- variable - lr * gradient
    return {k: np.array(v) for k, v in rec.items()}

In [ ]:
targets = [(1.0, 1.0), (-1.0, 1.0), (0.5, -1.5), (1.6, 0.2), (-1.2, -0.6), (0.0, 1.2)]
q_start = (0.5, 1.0)

rec = reach(arm_model, targets, q_start)

q_final = rec["q"][-1]
_, tip_real = forward_kinematics(q_final[:, 0], q_final[:, 1])            # where the arm really is
tip_net = predict(arm_model, torch.tensor(q_final, dtype=torch.float32))  # network's guess from q alone
tip_relaxed = rec["y_relaxed"][-1]                                        # output of the relaxed latents

print("target          real error   network error   relaxed error")
for i, t in enumerate(targets):
    print(f"{str(t):14s}  {np.linalg.norm(tip_real[i] - t):10.3f}   "
          f"{np.linalg.norm(tip_net[i] - t):13.3f}   {np.linalg.norm(tip_relaxed[i] - t):13.4f}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
_, tips_path = forward_kinematics(rec["q"][..., 0], rec["q"][..., 1])   # (frames, arms, 2)
for i, (ax, t) in enumerate(zip(axes.flat, targets)):
    arm_axes(ax, f"target {t}")
    draw_arm(ax, *rec["q"][0, i], color="gray", alpha=0.4, label="start")
    ax.plot(tips_path[:, i, 0], tips_path[:, i, 1], color="C0", lw=1, alpha=0.6)
    draw_arm(ax, *q_final[i], color="C0", label="final")
    ax.plot(*tip_relaxed[i], "o", color="C1", label="relaxed output")
    ax.plot(*t, "rx", ms=12, mew=2, label="target")
axes[0, 0].legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.show()

In [ ]:
_, tips_path = forward_kinematics(rec["q"][..., 0], rec["q"][..., 1])
real_error = np.linalg.norm(tips_path - np.array(targets), axis=2)   # (frames, arms)
energy = rec["E_hidden"] + rec["E_out"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, t in enumerate(targets):
    axes[0].semilogy(rec["step"], energy[:, i], label=str(t))
    axes[1].plot(rec["step"], real_error[:, i], label=str(t))
axes[0].set(xlabel="inference step", ylabel="energy E", title="energy (what inference minimises)")
axes[1].set(xlabel="inference step", ylabel="distance to target", title="real error (what we care about)")
axes[1].axhline(0.1, color="gray", ls="--")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.show()

In [ ]:
def save_reach_gif(rec, targets, path, fps=25):
    n = len(targets)
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.3 * ncols, 3.3 * nrows))
    axes = np.atleast_1d(axes).flat
    elbows, tips_path = forward_kinematics(rec["q"][..., 0], rec["q"][..., 1])

    arms, trails, dots = [], [], []
    for ax, t in zip(axes, targets):
        arm_axes(ax)
        ax.plot(*t, "rx", ms=10, mew=2)
        trails.append(ax.plot([], [], color="C0", lw=1, alpha=0.5)[0])
        arms.append(ax.plot([], [], "-o", color="C0", lw=3, ms=5)[0])
        dots.append(ax.plot([], [], "o", color="C1", ms=6)[0])
    title = fig.suptitle("")
    fig.tight_layout()

    def update(k):
        for i in range(n):
            arms[i].set_data([0, elbows[k, i, 0], tips_path[k, i, 0]],
                             [0, elbows[k, i, 1], tips_path[k, i, 1]])
            trails[i].set_data(tips_path[:k + 1, i, 0], tips_path[:k + 1, i, 1])
            dots[i].set_data([rec["y_relaxed"][k, i, 0]], [rec["y_relaxed"][k, i, 1]])
        title.set_text(f"inference step {rec['step'][k]}")
        return arms + trails + dots

    FuncAnimation(fig, update, frames=len(rec["step"])).save(path, writer=PillowWriter(fps=fps), dpi=70)
    plt.close(fig)


gif_path = RUN_DIR / "reaching.gif"
save_reach_gif(rec, targets, gif_path)
display(Image(filename=str(gif_path)))

In [ ]:
target = (-1.2, -0.6)
n_starts = 12
starts = rng.uniform(-np.pi, np.pi, size=(n_starts, 2))
rec_many = reach(arm_model, [target] * n_starts, starts)

q_end = rec_many["q"][-1]
_, tip_end = forward_kinematics(q_end[:, 0], q_end[:, 1])
error_end = np.linalg.norm(tip_end - np.array(target), axis=1)
outside = np.any(np.abs(rec_many["q"]) > np.pi, axis=2).any(axis=0)   # ever left [-π, π]?

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
arm_axes(axes[0], f"12 starts, one target {target}")
for i in range(n_starts):
    ok = error_end[i] < 0.2
    draw_arm(axes[0], *q_end[i], color="C2" if ok else "C3", alpha=0.7)
axes[0].plot(*target, "kx", ms=14, mew=3)

for i in range(n_starts):
    axes[1].plot(rec_many["step"], rec_many["q"][:, i, 1], color="C3" if outside[i] else "C0", alpha=0.7)
axes[1].axhline(np.pi, color="gray", ls="--")
axes[1].axhline(-np.pi, color="gray", ls="--")
axes[1].set(xlabel="inference step", ylabel="θ2", title="elbow angle; red = left the trained range")
axes[1].grid(alpha=0.3)
plt.show()

print("final errors:", np.round(error_end, 3))
print("reached (< 0.2):", int(np.sum(error_end < 0.2)), "of", n_starts,
      "  left the trained range:", int(outside.sum()))